In [34]:
using LowLevelFEM, LinearAlgebra

In [35]:
structured_box_mesh(n=40)

mat = Material("body");

In [36]:
probE = Problem([mat], type=:Solid)

stiffnessMatrix(probE)
GC.gc()
@time K0 = stiffnessMatrix(probE);

  5.806558 seconds (35.20 M allocations: 14.544 GiB, 16.65% gc time)


In [37]:
Pu = Problem([mat], type=:VectorField, dim=3, field=:u);

In [38]:
μ = mat.μ
λ = mat.λ
D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

GC.gc()
∫(SymGrad(Pu) ⋅ Dμ ⋅ SymGrad(Pu) + Div(Pu) ⋅ λ ⋅ Div(Pu));

In [39]:
GC.gc()
@time K1 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=1)
GC.gc()
@time K2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=2)
GC.gc()
@time K3 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=4);

  1.966815 seconds (641.34 k allocations: 433.326 MiB)
  1.448406 seconds (641.61 k allocations: 555.002 MiB)
  1.423420 seconds (641.71 k allocations: 798.331 MiB)


In [40]:
norm(K0.A - K1.A) / norm(K0.A)

2.241104209761531e-16

In [41]:
norm(K0.A - K2.A) / norm(K0.A)

2.242910000877906e-16

In [42]:
norm(K0.A - K3.A) / norm(K0.A)

2.242910000877906e-16

In [43]:
build_csc_pattern(Pu, Pu; Ω="body")
GC.gc()
@time Kpattern = build_csc_pattern(Pu, Pu; Ω="body")

  0.313328 seconds (641.22 k allocations: 426.803 MiB, 8.97% gc time)


206763×206763 SparseArrays.SparseMatrixCSC{Float64, Int64} with 15944049 stored entries:
⎡⡿⣯⡏⣿⡿⠽⠿⠿⢷⣷⣿⣛⣚⣚⣓⣓⣓⣛⣚⣚⣚⣓⣓⣷⣿⣮⣬⣬⣥⣥⡥⠭⠬⠬⠭⠥⠥⠭⠭⠽⎤
⎢⣯⣭⠿⣧⣿⢹⢸⡽⡇⡇⣿⢾⢸⢸⡇⡇⡇⣿⢸⢸⢸⡇⡇⡇⣿⢸⢸⢸⡏⡏⡏⣿⢹⢹⢽⡏⡏⣯⣿⢩⎥
⎢⣟⡏⣟⣛⡻⣮⣓⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠈⠈⠈⠁⠁⠁⠉⠈⠈⠈⠁⠁⠁⠹⣮⎥
⎢⣿⡇⣖⡶⠙⠘⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢽⣷⠭⠭⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢻⣻⣟⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣺⢸⣒⣒⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢽⢸⠭⠭⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣽⢸⣭⣭⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣺⢸⣒⣒⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢾⢸⠶⠶⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢽⣼⠭⠭⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡻⣿⣛⣛⡂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡂⣿⣒⣒⡂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠅⣿⡯⠭⠅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡅⡏⣯⣭⡅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡂⡇⣗⣒⡂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⎥
⎢⠇⡇⡷⠷⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⎥
⎢⡅⡇⡯⣭⠅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⎥
⎣⣇⡇⡟⣛⡳⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⎦

In [44]:
fill!(Kpattern.nzval, 0.0)
@time fill!(Kpattern.nzval, 0.0)
∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=1, csc_matrix=Kpattern)

fill!(Kpattern.nzval, 0.0)
@time KK1 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=1, csc_matrix=Kpattern)

fill!(Kpattern.nzval, 0.0)
@time KK2 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=2, csc_matrix=Kpattern)

fill!(Kpattern.nzval, 0.0)
@time KK3 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu), threads=4, csc_matrix=Kpattern)


  0.033922 seconds (1 allocation: 48 bytes)
  1.692336 seconds (134 allocations: 6.523 MiB)
  1.114010 seconds (276 allocations: 128.192 MiB)
  1.249920 seconds (500 allocations: 371.530 MiB, 0.12% gc time)


sparse([1, 2, 3, 139, 140, 141, 142, 143, 144, 1078  …  206643, 206644, 206645, 206646, 206758, 206759, 206760, 206761, 206762, 206763], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763], [1175.2136752136657, 400.64102564102234, -400.64102564102285, 267.09401709401766, 200.32051282051262, 80.12820512820237, 267.0940170940145, -80.1282051282041, -200.3205128205119, -534.1880341880327  …  -53.41880341880301, -3.979039320256561e-13, 1.2221335055073723e-12, 1068.3760683760836, 2.9416469260468148e-12, 6.536993168992922e-13, 1068.3760683760868, 4.774847184307873e-12, -1.0800249583553523e-12, 9401.709401709404], 206763, 206763)

In [45]:
norm(K0.A - KK1.A) / norm(K0.A)

2.242910000877906e-16

In [46]:
norm(K0.A - KK2.A) / norm(K0.A)

2.242910000877906e-16

In [47]:
norm(K0.A - KK3.A) / norm(K0.A)

2.242910000877906e-16